In [1]:
import sys

# Exception Handling for missing Scapy library
try:
    from scapy.all import sniff, TCP, Raw, IP
    SCAPY_AVAILABLE = True
except ImportError:
    SCAPY_AVAILABLE = False

def display_title():
    """Displays a clean title banner for the sniffing module."""
    print("\n" + "=" * 60)
    print("     NETWORK SNIFFER: CLEAR TEXT CREDENTIAL DETECTOR")
    print("=" * 60)

def packet_callback(packet):
    """Callback function executed for every captured packet to inspect payloads."""
    try:
        # Check if the packet has an IP layer, TCP layer, and Raw payload data
        if packet.haslayer(IP) and packet.haslayer(TCP) and packet.haslayer(Raw):
            payload = packet[Raw].load.decode('utf-8', errors='ignore').lower()
            
            # Common unencrypted login keywords found in HTTP POST requests, FTP, Telnet, etc.
            credential_keywords = ['user=', 'pass=', 'username=', 'password=', 'login=', 'auth=']
            
            for keyword in credential_keywords:
                if keyword in payload:
                    src_ip = packet[IP].src
                    dst_ip = packet[IP].dst
                    sport = packet[TCP].sport
                    dport = packet[TCP].dport
                    
                    print(f"\n[ALERT 🚨] Unencrypted Credential Pattern Found!")
                    print(f"  • Source      : {src_ip}:{sport}")
                    print(f"  • Destination : {dst_ip}:{dport}")
                    print(f"  • Matched Key : '{keyword}'")
                    print(f"  • Payload Snippet:\n    {payload.strip()[:200]}")
                    print("-" * 60)
                    break
    except Exception as e:
        # Suppress individual packet processing errors to keep the sniffer running
        pass

def run_sniffer():
    """Configures and runs the Scapy packet sniffer."""
    if not SCAPY_AVAILABLE:
        print("\n[EXCEPTION CAUGHT] ImportError: Scapy library is not installed.")
        print("[INFO] Please install it via terminal using: pip install scapy")
        return

    print("\n--- START PACKET SNIFFER ---")
    print("[INFO] Sniffing will inspect TCP packets for unencrypted text.")
    print("[INFO] Press Ctrl+C in this terminal window to stop sniffing at any time.\n")
    
    try:
        # Prompt user for packet count limit to prevent endless hanging
        count_input = input("Enter max number of packets to capture (e.g., 50, or type '0' for infinite): ").strip()
        packet_count = int(count_input) if count_input != '0' else 0
        
        print(f"\n[OUTPUT] Starting live network capture... Listening for traffic...\n")
        
        # Scapy sniff function (store=0 saves memory by not keeping packets in RAM)
        sniff(filter="tcp", prn=packet_callback, count=packet_count, store=0)
        
        print("\n[OUTPUT] Sniffing session completed.")
        
    except PermissionError:
        print("\n[EXCEPTION CAUGHT] PermissionError: Packet sniffing requires Administrator / Root privileges.")
        print("[INFO] Please run your terminal or script as Administrator (Windows) or using 'sudo' (Linux/macOS).")
    except ValueError:
        print("\n[EXCEPTION CAUGHT] ValueError: Please enter a valid integer for packet count.")
    except KeyboardInterrupt:
        print("\n\n[OUTPUT] Sniffing stopped manually by user.")
    except Exception as e:
        print(f"\n[EXCEPTION CAUGHT] An unexpected error occurred: {e}")

def main():
    """Main program controller managing options, flow, and loop controls."""
    while True:
        display_title()
        print("1. Start Sniffer (Detect Unencrypted Credentials)")
        print("2. Exit Program")
        
        choice = input("\nEnter your choice (1-2): ").strip()
        
        if choice == '1':
            run_sniffer()
        elif choice == '2':
            print("\n" + "=" * 60)
            print("  Exiting Network Sniffer. Goodbye!")
            print("=" * 60 + "\n")
            break
        else:
            print("\n[ERROR] Invalid choice. Please enter 1 or 2.")
            
        # Continue or Exit prompt
        cont = input("\nWould you like to return to the menu? (y/n): ").strip().lower()
        if cont != 'y':
            print("\n" + "=" * 60)
            print("  Exiting program. Goodbye!")
            print("=" * 60 + "\n")
            break

if __name__ == "__main__":
    main()


     NETWORK SNIFFER: CLEAR TEXT CREDENTIAL DETECTOR
1. Start Sniffer (Detect Unencrypted Credentials)
2. Exit Program



Enter your choice (1-2):  1



--- START PACKET SNIFFER ---
[INFO] Sniffing will inspect TCP packets for unencrypted text.
[INFO] Press Ctrl+C in this terminal window to stop sniffing at any time.



Enter max number of packets to capture (e.g., 50, or type '0' for infinite):  25



[OUTPUT] Starting live network capture... Listening for traffic...


[OUTPUT] Sniffing session completed.



Would you like to return to the menu? (y/n):  y



     NETWORK SNIFFER: CLEAR TEXT CREDENTIAL DETECTOR
1. Start Sniffer (Detect Unencrypted Credentials)
2. Exit Program



Enter your choice (1-2):  2



  Exiting Network Sniffer. Goodbye!

